# 🔍 Resume Screening System
**Future Interns — ML Internship Track, Task 3 (2026)**  
**Author:** Lamla Mhlana  

---

## Objective
Build an ML system that:
1. Reads and preprocesses resume text
2. Extracts skills using NLP
3. Compares resumes to a job description
4. Ranks candidates by role fit
5. Highlights skill gaps

**Dataset:** [Kaggle Resume Dataset](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset) — 2,484 resumes across 25 categories

## 1. Setup & Imports

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from collections import Counter

from src.preprocessor import basic_clean, deep_clean, preprocess_resume
from src.skill_extractor import (
    extract_skills_from_text, get_flat_skill_set,
    compute_skill_gap, skill_match_score
)
from src.scorer import rank_candidates, get_top_n, score_summary
from src.skill_taxonomy import SKILL_TAXONOMY

plt.style.use('seaborn-v0_8-darkgrid')
print('✅ All modules loaded')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/resume_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# Standardise column names
df = df.rename(columns={
    'Resume_str': 'resume_text',
    'Category': 'category'
})

# Drop rows with missing resume text
df = df.dropna(subset=['resume_text'])
df['resume_text'] = df['resume_text'].astype(str)

# Add a candidate name column (simulated for anonymised data)
df['candidate_name'] = ['Candidate_' + str(i+1).zfill(4) for i in range(len(df))]

print(f'Clean dataset: {len(df)} resumes across {df["category"].nunique()} categories')
df[['candidate_name', 'category', 'resume_text']].head()

## 3. Exploratory Data Analysis

In [ ]:
# Resumes per category
cat_counts = df['category'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(cat_counts.index, cat_counts.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of Resumes')
ax.set_title('Resume Distribution by Job Category', fontsize=14, fontweight='bold')
ax.bar_label(bars, padding=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Resume length distribution
df['text_length'] = df['resume_text'].apply(len)
df['word_count'] = df['resume_text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['text_length'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Characters')

axes[1].hist(df['word_count'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')

plt.suptitle('Resume Length Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

print(df[['text_length', 'word_count']].describe().round(1))

## 4. Text Preprocessing

In [ ]:
# Show preprocessing transformation on one sample
sample = df['resume_text'].iloc[0]
print('=== RAW (first 400 chars) ===')
print(sample[:400])
print()
print('=== BASIC CLEAN ===')
print(basic_clean(sample)[:400])
print()
print('=== DEEP CLEAN (vectorizable) ===')
print(deep_clean(sample)[:400])

In [ ]:
# Apply basic cleaning to full dataset
print('Cleaning resumes...')
df['cleaned_text'] = df['resume_text'].apply(basic_clean)
print('✅ Done')

## 5. Skill Extraction

In [ ]:
# Extract skills for a single resume
sample_skills = extract_skills_from_text(df['resume_text'].iloc[0])
for category, skills in sample_skills.items():
    print(f'{category:30s}: {skills}')

In [ ]:
# Most common skills across entire dataset
all_skills_found = []
for text in df['resume_text']:
    all_skills_found.extend(get_flat_skill_set(text))

skill_freq = Counter(all_skills_found).most_common(20)
skills_df = pd.DataFrame(skill_freq, columns=['skill', 'count'])

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(skills_df['skill'][::-1], skills_df['count'][::-1], color='mediumseagreen')
ax.set_xlabel('Frequency across all resumes')
ax.set_title('Top 20 Most Common Skills in Dataset', fontsize=14, fontweight='bold')
ax.bar_label(bars, padding=3, fontsize=9)
plt.tight_layout()
plt.show()

## 6. Define Job Description & Screen Candidates

In [ ]:
JOB_DESCRIPTION = """
We are looking for a Data Engineer to join our analytics team.

Requirements:
- 2+ years of experience in data engineering or a related field
- Strong proficiency in Python and SQL
- Experience with Apache Airflow, dbt, or similar pipeline tools
- Familiarity with cloud platforms: AWS, GCP, or Azure
- Experience with PostgreSQL, BigQuery, or Snowflake
- Knowledge of Apache Spark or Kafka is a plus
- Ability to build and maintain ETL/ELT pipelines
- Understanding of data warehousing concepts
- Experience with Docker and basic Linux/Unix environments
- Good communication and teamwork skills
"""

print('Job Description loaded.')
print(f'Required skills identified: {sorted(get_flat_skill_set(JOB_DESCRIPTION))}')

In [ ]:
# Use a subset for faster demo (remove [:100] to run on full dataset)
sample_df = df[['candidate_name', 'category', 'resume_text']].copy().iloc[:100]

print(f'Screening {len(sample_df)} candidates...')
ranked_df = rank_candidates(sample_df, JOB_DESCRIPTION)
print('✅ Ranking complete')
ranked_df[['rank', 'candidate_name', 'category', 'composite_score',
           'skill_score', 'tfidf_score', 'experience_score']].head(10)

## 7. Candidate Ranking & Visualisation

In [ ]:
# Score distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color, label in zip(
    axes,
    ['composite_score', 'skill_score', 'tfidf_score'],
    ['steelblue', 'mediumseagreen', 'coral'],
    ['Composite Score', 'Skill Match Score', 'TF-IDF Similarity']
):
    ax.hist(ranked_df[col], bins=20, color=color, edgecolor='white')
    ax.set_title(label)
    ax.set_xlabel('Score (0–100)')
    ax.axvline(ranked_df[col].mean(), color='black', linestyle='--', label='Mean')
    ax.legend()

plt.suptitle('Score Distributions Across All Screened Candidates', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 candidates — horizontal bar chart
top10 = get_top_n(ranked_df, 10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    top10['candidate_name'][::-1],
    top10['composite_score'][::-1],
    color=plt.cm.RdYlGn(top10['composite_score'][::-1] / 100)
)
ax.set_xlabel('Composite Score (0–100)')
ax.set_title('Top 10 Ranked Candidates', fontsize=14, fontweight='bold')
ax.axvline(60, color='red', linestyle='--', alpha=0.7, label='Shortlist threshold (60)')
ax.legend()
ax.bar_label(bars, fmt='%.1f', padding=3)
plt.tight_layout()
plt.show()

## 8. Skill Gap Analysis

In [ ]:
# Detailed gap report for the top candidate
top_candidate = ranked_df.iloc[0]

print(f"=== Skill Gap Report: {top_candidate['candidate_name']} ===")
print(f"Composite Score : {top_candidate['composite_score']}/100")
print(f"Skill Match     : {top_candidate['skill_score']}/100")
print(f"TF-IDF Sim.     : {top_candidate['tfidf_score']}/100")
print(f"Experience      : {top_candidate['experience_score']}/100")
print()
print(f"✅ Matched Skills : {top_candidate['matched_skills']}")
print(f"❌ Missing Skills : {top_candidate['missing_skills']}")
print(f"➕ Extra Skills   : {top_candidate['extra_skills'][:10]}")

In [ ]:
# Most common missing skills across all candidates
all_missing = []
for _, row in ranked_df.iterrows():
    all_missing.extend(row['missing_skills'])

missing_freq = Counter(all_missing).most_common(15)
miss_df = pd.DataFrame(missing_freq, columns=['skill', 'count'])

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(miss_df['skill'], miss_df['count'], color='tomato', edgecolor='white')
ax.set_xticklabels(miss_df['skill'], rotation=40, ha='right')
ax.set_ylabel('Candidates Missing This Skill')
ax.set_title('Most Commonly Missing Skills (vs Job Description)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Summary Statistics

In [ ]:
summary = score_summary(ranked_df)
shortlisted = ranked_df[ranked_df['composite_score'] >= summary['shortlist_threshold']]

print('=== Screening Summary ===')
for k, v in summary.items():
    print(f'{k:25s}: {v}')
print(f'{"shortlisted_count":25s}: {len(shortlisted)}')
print(f'{"shortlist_rate":25s}: {round(len(shortlisted)/len(ranked_df)*100,1)}%')

## 10. Export Results

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

export_cols = [
    'rank', 'candidate_name', 'category',
    'composite_score', 'skill_score', 'tfidf_score', 'experience_score',
    'matched_skills', 'missing_skills'
]

ranked_df[export_cols].to_csv('../data/ranked_candidates.csv', index=False)
print('✅ Exported: data/ranked_candidates.csv')